# Streaming - Simulador de Eventos

## Objetivo

Este notebook simula a chegada de novas medições do Indicador Criança Alfabetizada em tempo quase real.

Cada registro representa um evento produzido por um sistema externo contendo uma nova medição de alfabetização para determinado nível geográfico.

O simulador desempenha o papel de um **Producer de eventos**.

No pipeline de Streaming, esses eventos serão posteriormente consumidos pelo Spark Structured Streaming e processados pelas camadas Bronze, Silver e Gold.

Os eventos deste notebook são **dados sintéticos utilizados exclusivamente para demonstração técnica do pipeline**, não representando resultados oficiais do INEP ou da Base dos Dados.

# Configuração

## Bibliotecas

In [0]:
import uuid

from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

import time

# Estrutura da Simulação

## Schema de simulação

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS simulacao
""")

## Tabela de Entrada

A tabela `simulacao.eventos_alfabetizacao` funciona como uma área de entrada
(*landing zone*) para os eventos sintéticos.

O simulador adiciona novos registros utilizando escrita em modo `append`,
preservando os eventos anteriormente publicados.

Esta tabela representa a origem que será posteriormente consumida pelo
Spark Structured Streaming.

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS simulacao.eventos_alfabetizacao (
    event_id STRING,
    event_timestamp TIMESTAMP,
    tipo_evento STRING,
    ano INT,
    nivel_geografico STRING,
    id_geografia STRING,
    rede STRING,
    taxa_alfabetizacao DOUBLE,
    percentual_participacao DOUBLE,
    origem_evento STRING
)
USING DELTA
""")

## Schema dos eventos

In [0]:
schema_evento = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("tipo_evento", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("nivel_geografico", StringType(), False),
    StructField("id_geografia", StringType(), False),
    StructField("rede", StringType(), False),
    StructField("taxa_alfabetizacao", DoubleType(), False),
    StructField("percentual_participacao", DoubleType(), True),
    StructField("origem_evento", StringType(), False)
])

# Definição dos Eventos

## Função de Publicação

In [0]:
def publicar_evento(evento):

    registro = {
        "event_id": str(uuid.uuid4()),
        "event_timestamp": datetime.now(timezone.utc),
        "tipo_evento": "NOVA_MEDICAO_DESEMPENHO",

        "ano": evento["ano"],
        "nivel_geografico": evento["nivel_geografico"],
        "id_geografia": evento["id_geografia"],
        "rede": evento["rede"],

        "taxa_alfabetizacao": evento["taxa_alfabetizacao"],
        "percentual_participacao": evento["percentual_participacao"],

        "origem_evento": "SIMULACAO_TECH_CHALLENGE"
    }

    df_evento = spark.createDataFrame(
        [registro],
        schema=schema_evento
    )

    (
        df_evento.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "simulacao.eventos_alfabetizacao"
        )
    )

    print(
        f"Evento publicado | "
        f"{registro['nivel_geografico']} "
        f"{registro['id_geografia']} | "
        f"Taxa: {registro['taxa_alfabetizacao']}%"
    )

## Eventos Simulados

In [0]:
eventos_simulados = [
    {
        "ano": 2026,
        "nivel_geografico": "UF",
        "id_geografia": "BA",
        "rede": "Pública",
        "taxa_alfabetizacao": 58.5,
        "percentual_participacao": 90.0
    },
    {
        "ano": 2026,
        "nivel_geografico": "UF",
        "id_geografia": "RN",
        "rede": "Pública",
        "taxa_alfabetizacao": 53.0,
        "percentual_participacao": 86.0
    },
    {
        "ano": 2026,
        "nivel_geografico": "MUNICIPIO",
        "id_geografia": "2406908",
        "rede": "Municipal",
        "taxa_alfabetizacao": 28.0,
        "percentual_participacao": 88.0
    },
    {
        "ano": 2026,
        "nivel_geografico": "MUNICIPIO",
        "id_geografia": "1718501",
        "rede": "Municipal",
        "taxa_alfabetizacao": 22.0,
        "percentual_participacao": 91.0
    },
    {
        "ano": 2026,
        "nivel_geografico": "BRASIL",
        "id_geografia": "BRASIL",
        "rede": "Pública",
        "taxa_alfabetizacao": 68.5,
        "percentual_participacao": 89.5
    }
]

# Execução da Simulação

## Publicação Sequencial

Os eventos são publicados individualmente com um intervalo de cinco segundos, simulando a chegada contínua de novas medições em tempo quase real.

O intervalo reduzido é utilizado apenas para fins de demonstração do pipeline.

In [0]:
for evento in eventos_simulados:

    publicar_evento(evento)

    time.sleep(5)

# Validação dos Resultados

In [0]:
df_eventos = spark.table(
    "simulacao.eventos_alfabetizacao"
)

print(
    f"Quantidade de eventos disponíveis: "
    f"{df_eventos.count()}"
)

display(
    df_eventos
    .orderBy("event_timestamp")
)

# Teste Posterior - Inserindo novos Eventos

In [0]:
novo_evento = {
    "ano": 2026,
    "nivel_geografico": "UF",
    "id_geografia": "BA",
    "rede": "Pública",
    "taxa_alfabetizacao": 62.0,
    "percentual_participacao": 93.0
}

publicar_evento(novo_evento)

In [0]:
novo_evento = {
    "ano": 2026,
    "nivel_geografico": "UF",
    "id_geografia": "BA",
    "rede": "Pública",
    "taxa_alfabetizacao": 26.0,
    "percentual_participacao": 98.0
}

publicar_evento(novo_evento)